# 04 — Genotypic Extraction: Phased Short-Read WGS
## NIH All of Us — Sickle Cell Disease (SCD) Analysis

| Field | Details |
|-------|---------|
| **Project** | Sickle Cell Disease Multi-Modal Analysis |
| **Dataset** | All of Us Controlled Tier Dataset v8 — Short-Read WGS (srWGS), phased |
| **Description** | Extracts phased genotypes for 16 disease-relevant SNPs across 6 genes from the All of Us phased short-read WGS dataset for 476 SCD participants |
| **Input** | All of Us phased WGS VCFs (`fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing`), `scd_participants_476.txt` |
| **Output** | `FINAL_MERGED_ALL_476_16pos.phased.vcf.gz`, `ALL_476_genotypes_long.csv`, `ALL_variants_summary_with_haplotypes_and_genes.csv` |

---

## Target Genes & SNPs

| Gene | Chromosome | N SNPs | Biological Relevance |
|------|-----------|--------|---------------------|
| **HBB-cluster** | chr11 | 3 | Beta-globin mutations — direct SCD causal variants |
| **HBG2** | chr11 | 1 | Foetal haemoglobin (HbF) — protective modifier |
| **BCL11A** | chr2 | 3 | HbF regulatory locus — major SCD modifier gene |
| **HBS1L-MYB (HMIP)** | chr6 | 3 | HbF quantitative trait locus |
| **TNF** | chr6 | 1 | Inflammatory cytokine — VOC severity modifier |
| **APOE** | chr19 | 2 | Lipid metabolism — stroke risk modifier |
| **APOL1** | chr22 | 2 | Kidney disease risk in SCD |
| **VCAM1** | chr1 | 1 | Vascular adhesion — VOC pathophysiology |

---

## Workflow
1. Define 16 target SNP positions and SCD participant list  
2. Use Hail to extract phased genotypes from All of Us WGS VCFs per chromosome  
3. Download per-chromosome VCF files to local workspace  
4. Use bcftools to merge, deduplicate and validate all chromosomes  
5. Build wide and long genotype tables per participant  
6. Map variants to genes and summarise allele frequencies  
7. Visualise genotype distributions across the SCD cohort  

---
> ⚠️ **All of Us Compliance:** This notebook must be run inside the All of Us Researcher Workbench.  
> Requires **Hail** and **bcftools** (pre-installed in All of Us Jupyter environment).  
> No participant data is stored in this repository.

---
## ⚙️ Before You Begin

> **Prerequisite:** Notebook 01 must have been run first. You need  
> `scd_participants_476.txt` uploaded to your workspace bucket inputs folder.

### Additional requirements for this notebook
- **Hail** — pre-installed in All of Us Jupyter (`import hail as hl`)
- **bcftools** — pre-installed in All of Us Jupyter (`!bcftools --version`)
- **Requester Pays access** — your workspace must have `GOOGLE_PROJECT` env variable set
- Access to `fc-aou-datasets-controlled` bucket (Controlled Tier only)

### Environment variables used
```python
WORKSPACE_BUCKET  # your personal workspace GCS bucket
GOOGLE_PROJECT    # your Terra/All of Us project ID
```

### Run order
1. Select **Python 3** kernel  
2. Run **Kernel → Restart & Run All**  
   *OR* run cells one by one with **Shift + Enter**  
> ⏱️ *Hail cells may take 5–20 minutes each due to WGS data size.*

---

## Step 1 — Define Target SNPs and Gene Map

**What this does:**  
Creates the reference dictionary mapping each (chromosome, position) pair  
to its gene name. This is the master lookup table used throughout the notebook.

**16 SNPs across 8 disease-relevant genes:**
- `HBB-cluster` (chr11) — beta-globin mutations directly causing SCD  
- `HBG2` (chr11) — foetal haemoglobin regulator  
- `BCL11A` (chr2) — major HbF modifier locus  
- `HBS1L-MYB/HMIP` (chr6) — HbF quantitative trait locus  
- `TNF` (chr6) — inflammatory severity modifier  
- `APOE` (chr19) — stroke/lipid risk modifier  
- `APOL1` (chr22) — kidney disease risk  
- `VCAM1` (chr1) — vascular adhesion / VOC pathophysiology

In [ ]:
# -------------------
# 13 SNPs present in ACAF (Kyera checked your list)
# HMIP chr6:135097495, 135097526, 135097880 are excluded here
# -------------------
acaf_pos_to_gene = {
    ("chr19", 44908684): "APOE",
    ("chr19", 44908822): "APOE",
    ("chr22", 36265860): "APOL1",
    ("chr22", 36265988): "APOL1",
    ("chr2",  60487726): "BCL11A",
    ("chr2",  60490908): "BCL11A",
    ("chr2",  60498316): "BCL11A",
    ("chr11", 5239228):  "HBB-cluster",
    ("chr11", 5242453):  "HBB-cluster",
    ("chr11", 5248569):  "HBB-cluster",
    ("chr11", 5254939):  "HBG2",
    ("chr6",  31575254): "TNF",
    ("chr1", 100718269): "VCAM1",
}

# Use the same reference genome as the ACAF MT
rg = acaf_mt_scd.locus.dtype.reference_genome

# Build loci for these 13 positions
acaf_loci_list = [
    hl.parse_locus(f"{chrom}:{pos}", reference_genome=rg)
    for (chrom, pos) in acaf_pos_to_gene.keys()
]
acaf_loci_set = hl.set(acaf_loci_list)

# Subset ACAF MT (476 SCD samples × 13 loci)
acaf_mt_scd_13 = acaf_mt_scd.filter_rows(acaf_loci_set.contains(acaf_mt_scd.locus))

print("Done 13-SNP ACAF filter.")
print("Cols (samples):", acaf_mt_scd_13.count_cols())


---
## Step 2 — Initialise Hail and Load ACAF Threshold Matrix Table

**What this does:**  
Imports Hail and sets up paths to the All of Us ACAF (Allele Count/Allele Frequency)  
threshold Matrix Table — the main WGS data store.

**Key paths:**
- `ACAF_MT_PATH` — Hail Matrix Table for all ~245k All of Us WGS samples  
- `path_1col` — your 476 SCD participant IDs file in the workspace bucket

> ⏱️ *Hail initialisation may take 2–3 minutes on first run.*

In [ ]:
import hail as hl
import pandas as pd

# -------------------
# Paths
# -------------------
BUCKET = "${WORKSPACE_BUCKET}"
path_1col = f"{BUCKET}/inputs/scd_participants_iid.txt"

CDR_STORAGE_PATH = "gs://fc-aou-datasets-controlled/v8"
ACAF_MT_PATH = f"{CDR_STORAGE_PATH}/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"

# -------------------
# Init Hail
# -------------------
hl.init(app_name="SCD_476_16SNPs_ACAF", default_reference="GRCh38")

# -------------------
# Load ACAF Threshold MT (Kyera's path)
# -------------------
acaf_mt = hl.read_matrix_table(ACAF_MT_PATH)
print("ACAF col key:", acaf_mt.col_key)

# -------------------
# Filter to your 476 SCD participants
# -------------------
keep_ht = hl.import_table(path_1col, no_header=True)
keep_ht = keep_ht.rename({'f0': 's'}).key_by('s')

acaf_mt_scd = acaf_mt.filter_cols(hl.is_defined(keep_ht[acaf_mt.s]))
print("Cols after SCD filter:", acaf_mt_scd.count_cols())  # should be 476

# -------------------
# 13 SNPs present in ACAF (HMIP chr6:135097495/526/880 excluded)
# -------------------
acaf_pos_to_gene = {
    ("chr19", 44908684): "APOE",
    ("chr19", 44908822): "APOE",
    ("chr22", 36265860): "APOL1",
    ("chr22", 36265988): "APOL1",
    ("chr2",  60487726): "BCL11A",
    ("chr2",  60490908): "BCL11A",
    ("chr2",  60498316): "BCL11A",
    ("chr11", 5239228):  "HBB-cluster",
    ("chr11", 5242453):  "HBB-cluster",
    ("chr11", 5248569):  "HBB-cluster",
    ("chr11", 5254939):  "HBG2",
    ("chr6",  31575254): "TNF",
    ("chr1", 100718269): "VCAM1",
}

rg = acaf_mt_scd.locus.dtype.reference_genome

acaf_loci_list = [
    hl.parse_locus(f"{chrom}:{pos}", reference_genome=rg)
    for (chrom, pos) in acaf_pos_to_gene.keys()
]
acaf_loci_set = hl.set(acaf_loci_list)

# -------------------
# Filter ACAF MT (SCD-only) to these 13 SNPs
# -------------------
acaf_mt_scd_13 = acaf_mt_scd.filter_rows(acaf_loci_set.contains(acaf_mt_scd.locus))
print("Done 13-SNP ACAF filter.")
print("Cols (samples):", acaf_mt_scd_13.count_cols())

entry_schema = acaf_mt_scd_13.entry
print(entry_schema)

# If field is called GT:
example = acaf_mt_scd_13.entry.take(1)
print(example)

# -------------------
# Export phased VCF: 476 samples × 13 ACAF SNPs
# -------------------
acaf_mt_export = acaf_mt_scd_13.naive_coalesce(64)

acaf_vcf = f"{BUCKET}/outputs/scd_476_acaf_13snps_phased.vcf.bgz"
hl.export_vcf(acaf_mt_export, acaf_vcf)

print("Exported ACAF 13-SNP VCF to:", acaf_vcf)


---
## Step 3 — Set Up Workspace Environment Variables

**What this does:**  
Loads your All of Us workspace environment variables and verifies they are set correctly.

**Expected output:** Your workspace bucket path and Google project ID printed.

In [ ]:
import os

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT = os.environ["GOOGLE_PROJECT"]

BUCKET = "${WORKSPACE_BUCKET}"
SAMPLES_GS = f"{BUCKET}/inputs/scd_participants_iid.txt"

PHASE_DIR = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"

print("GOOGLE_PROJECT:", GOOGLE_PROJECT)
print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET)
print("SAMPLES_GS:", SAMPLES_GS)
print("PHASE_DIR:", PHASE_DIR)


---
## Step 4 — Download SCD Participant List

**What this does:**  
Copies your 476 SCD participant ID file from the workspace bucket  
to the local Jupyter environment, then confirms the count.

**Expected output:** `476` printed (number of participants in the file).

In [ ]:
!gsutil cp {SAMPLES_GS} ./scd_participants_476.txt
!wc -l scd_participants_476.txt
!head -n 5 scd_participants_476.txt


---
## Step 5 — Define SNP Region Files

**What this does:**  
Creates two region list files used to subset the WGS VCFs:
- `snps_16_regions.txt` — the 16 final SNP positions  
- `snps_17_regions.txt` — 17 positions (includes chr22:36265996 for validation)

These files are passed to bcftools and Hail for targeted extraction.

**Format:** `chr:start-end` (one SNP per line, start = end for single-position SNPs)

In [ ]:
%%bash
cat > snps_16_regions.txt << 'EOF'
chr1:100718269-100718269
chr2:60487726-60487726
chr2:60490908-60490908
chr2:60498316-60498316
chr6:31575254-31575254
chr6:135097495-135097495
chr6:135097526-135097526
chr6:135097880-135097880
chr11:5239228-5239228
chr11:5242453-5242453
chr11:5248569-5248569
chr11:5254939-5254939
chr19:44908684-44908684
chr19:44908822-44908822
chr22:36265860-36265860
chr22:36265988-36265988
chr22:36265996-36265996
EOF

wc -l snps_16_regions.txt


In [ ]:
!bcftools --version | head -n 2


In [ ]:
!mkdir -p phased_extract


In [ ]:
%%bash
cat > snps_17_regions.txt <<'EOF'
chr1:100718269-100718269
chr2:60487726-60487726
chr2:60490908-60490908
chr2:60498316-60498316
chr6:31575254-31575254
chr6:135097495-135097495
chr6:135097526-135097526
chr6:135097880-135097880
chr11:5239228-5239228
chr11:5242453-5242453
chr11:5248569-5248569
chr11:5254939-5254939
chr19:44908684-44908684
chr19:44908822-44908822
chr22:36265860-36265860
chr22:36265988-36265988
chr22:36265996-36265996
EOF

wc -l snps_17_regions.txt
tail -n 5 snps_17_regions.txt


---
## Step 6 — Extract Phased Genotypes Using Hail (All Chromosomes)

**What this does:**  
For each target chromosome, uses Hail to:
1. Read the All of Us phased WGS VCF for that chromosome  
2. Filter to only your 476 SCD participants  
3. Filter to only the target SNP positions  
4. Export as a compressed, indexed VCF (`.vcf.bgz`) to your workspace bucket  

**Chromosomes processed:** chr1, chr2, chr6, chr11, chr19, chr22

> ⏱️ *Each chromosome extraction takes approximately 5–15 minutes.*  
> *Total runtime for all chromosomes: ~1–2 hours.*

In [ ]:
import os, hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]

PHASE_DIR = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUT = f"{WORKSPACE_BUCKET}/outputs/phased_17pos_476"

hl.init(
    default_reference="GRCh38",
    tmp_dir=f"{WORKSPACE_BUCKET}/hail_tmp",
    log=f"{WORKSPACE_BUCKET}/logs/hail_subset_17pos_476.log",
    idempotent=True,
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, ["fc-aou-datasets-controlled"]),
)

keep_path = f"{WORKSPACE_BUCKET}/inputs/scd_participants_iid.txt"
keep_ht = hl.import_table(keep_path, no_header=True).rename({'f0': 's'}).key_by('s')

chrom = "chr22"
positions = [36265860, 36265988, 36265996]  # ✅ include missing one

target_loci = hl.literal({hl.parse_locus(f"{chrom}:{p}", reference_genome="GRCh38") for p in positions})

vcf = f"{PHASE_DIR}/{chrom}_AOU_v8.2_allsamples_phased.vcf.gz"
mt = hl.import_vcf(vcf, reference_genome="GRCh38", force_bgz=True, min_partitions=64)

mt = mt.filter_rows(target_loci.contains(mt.locus))
mt = mt.filter_cols(hl.is_defined(keep_ht[mt.s]))
mt = mt.select_entries("GT")

print("rows:", mt.count_rows(), "cols:", mt.count_cols())

hl.export_vcf(mt, f"{OUT}/{chrom}.476.3pos.phased.vcf.bgz", tabix=True)
print("Wrote:", f"{OUT}/{chrom}.476.3pos.phased.vcf.bgz")


In [ ]:
%%bash
set -euo pipefail

OUT="${WORKSPACE_BUCKET}/outputs/phased_17pos_476/chr22.476.3pos.phased.vcf.bgz"

# download tiny file + index
gsutil cp "$OUT" ./chr22.476.3pos.vcf.bgz
gsutil cp "$OUT.tbi" ./chr22.476.3pos.vcf.bgz.tbi

# check positions (must be 3)
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' ./chr22.476.3pos.vcf.bgz


---
## Step 7 — Inspect Local VCF Files

**What this does:**  
Lists the local VCF files that have been downloaded and  
checks their sizes to confirm successful extraction.

In [ ]:
%%bash
ls -lh *.vcf.gz *.vcf.bgz 2>/dev/null | head


In [ ]:
%%bash
ls -lh *.vcf.gz *.vcf.bgz 2>/dev/null | sed -n '1,80p'


In [ ]:
%%bash
set -euo pipefail
find . -maxdepth 1 -type f \( -name "*.vcf.gz" -o -name "*.vcf.bgz" -o -name "*.vcf.bgz.tbi" -o -name "*.vcf.gz.tbi" \) -print -exec ls -lh {} \;


---
## Step 8 — Download Per-Chromosome VCFs from Bucket

**What this does:**  
Downloads the per-chromosome phased VCF files (16pos set) from  
your workspace bucket to the local Jupyter environment for bcftools processing.

**Files downloaded:**
- `chr1.476.1pos.phased.vcf.bgz`
- `chr2.476.3pos.phased.vcf.bgz`
- `chr6.476.4pos.phased.vcf.bgz`
- `chr11.476.4pos.phased.vcf.bgz`
- `chr19.476.2pos.phased.vcf.bgz`
- `chr22.476.3pos.phased.vcf.bgz`

In [ ]:
%%bash
set -euo pipefail
gsutil ls "${WORKSPACE_BUCKET}/outputs/phased_16pos_476/"


In [ ]:
%%bash
set -euo pipefail

BASE="${WORKSPACE_BUCKET}/outputs/phased_16pos_476"

# download other chromosomes (exclude chr22)
gsutil -m cp "${BASE}/chr1"*.vcf.bgz .
gsutil -m cp "${BASE}/chr1"*.vcf.bgz.tbi .
gsutil -m cp "${BASE}/chr2"*.vcf.bgz .
gsutil -m cp "${BASE}/chr2"*.vcf.bgz.tbi .
gsutil -m cp "${BASE}/chr6"*.vcf.bgz .
gsutil -m cp "${BASE}/chr6"*.vcf.bgz.tbi .
gsutil -m cp "${BASE}/chr11"*.vcf.bgz .
gsutil -m cp "${BASE}/chr11"*.vcf.bgz.tbi .
gsutil -m cp "${BASE}/chr19"*.vcf.bgz .
gsutil -m cp "${BASE}/chr19"*.vcf.bgz.tbi .

# merge all + your fixed chr22
bcftools concat -a -Oz -o merged.476.fixed_chr22.vcf.gz \
  chr1*.vcf.bgz chr2*.vcf.bgz chr6*.vcf.bgz chr11*.vcf.bgz chr19*.vcf.bgz chr22.476.3pos.vcf.bgz

bcftools index -t merged.476.fixed_chr22.vcf.gz

# checks
echo "Samples:";  bcftools query -l merged.476.fixed_chr22.vcf.gz | wc -l
echo "Variants:"; bcftools view -H merged.476.fixed_chr22.vcf.gz | wc -l
echo "chr22:";    bcftools query -f '%CHROM\t%POS\n' -r chr22 merged.476.fixed_chr22.vcf.gz


In [ ]:
%%bash
set -euo pipefail

# show any chr22 files you have locally
ls -lh chr22* || true

# remove the OLD chr22 files (2pos) and any extra chr22 copies
rm -f chr22.476.2pos.phased.vcf.bgz chr22.476.2pos.phased.vcf.bgz.tbi

# confirm only the fixed one remains
ls -lh chr22* || true


In [ ]:
%%bash
set -euo pipefail

# delete the old chr22 (2pos)
rm -f chr22.476.2pos.phased.vcf.bgz chr22.476.2pos.phased.vcf.bgz.tbi

# de-duplicate exact duplicate files (safe)
# (if the file is truly duplicated, it will be the same path; this just re-lists)
ls -lh chr22* || true


---
## Step 9 — Re-extract with Hail (Full Per-Chromosome Loop)

**What this does:**  
Runs the complete Hail extraction loop across all target chromosomes  
using the 16-position region list. This is the primary extraction step  
if per-chromosome files do not yet exist in your bucket.

> ⏱️ *This is the longest-running step — allow 1–2 hours for all chromosomes.*

In [ ]:
%%bash
set -euo pipefail
PROJECT="terra-vpc-sc-ce41bb61"
PHASE_DIR="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"

for CHR in chr1 chr2 chr6 chr11 chr19 chr22; do
  echo "Downloading ${CHR} VCF + TBI..."
  gsutil -u "${PROJECT}" -m cp \
    "${PHASE_DIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz" \
    "${PHASE_DIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz.tbi" \
    phasing_vcfs/
done

ls -lh phasing_vcfs | head


---
## Step 10 — Fix Chr22 and Merge Into Single VCF

**What this does:**  
Merges the individual chromosome VCF files into a single merged VCF using  
`bcftools concat`, then verifies the chr22 position is correct.

**Expected output:** Merged VCF with all 476 samples and 16 variants.

In [ ]:
%%bash
set -euo pipefail

rm -f merged.476.fixed_chr22.vcf.gz merged.476.fixed_chr22.vcf.gz.tbi

bcftools concat -a -Oz -o merged.476.fixed_chr22.vcf.gz \
  chr1*.vcf.bgz chr2*.vcf.bgz chr6*.vcf.bgz chr11*.vcf.bgz chr19*.vcf.bgz chr22.476.3pos.vcf.bgz

bcftools index -t merged.476.fixed_chr22.vcf.gz

echo "Samples:";  bcftools query -l merged.476.fixed_chr22.vcf.gz | wc -l
echo "Variants:"; bcftools view -H merged.476.fixed_chr22.vcf.gz | wc -l
echo "chr22:";    bcftools query -f '%CHROM\t%POS\n' -r chr22 merged.476.fixed_chr22.vcf.gz


In [ ]:
%%bash
set -euo pipefail
VCF="merged.476.fixed_chr22.vcf.gz"   # change if your final file name differs

bcftools view -H -r chr22:36265996-36265996 "$VCF"


In [ ]:
%%bash
set -euo pipefail
VCF="merged.476.fixed_chr22.vcf.gz"

bcftools query \
  -f '%CHROM\t%POS\t%REF\t%ALT\t[%SAMPLE\t%GT\t%TGT\n]' \
  "$VCF" > phased_GT_all17_perSample.tsv

# preview
head -n 30 phased_GT_all17_perSample.tsv


In [ ]:
%%bash
set -euo pipefail
VCF="merged.476.fixed_chr22.vcf.gz"

# make a list of 10 sample IDs
bcftools query -l "$VCF" | head -n 10 > s10.txt

# print phased genotypes for these 10 samples across all variants
bcftools query -S s10.txt \
  -f '%CHROM\t%POS\t%REF\t%ALT[\t%SAMPLE=%GT]\n' \
  "$VCF" | head -n 30


In [ ]:
%%bash
df -h
du -sh phasing_vcfs 2>/dev/null || true
ls -lh phasing_vcfs 2>/dev/null | head || true


---
## Step 11 — Download Phasing VCFs for All Chromosomes

**What this does:**  
Downloads the remaining chromosome phasing VCF files to the  
`phasing_vcfs/` local directory for the final merge step.

In [ ]:
%%bash
set -euo pipefail

PROJECT="terra-vpc-sc-ce41bb61"
PHASE_DIR="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUTDIR="phased_extract_16"
mkdir -p "$OUTDIR"

for CHR in chr1 chr2 chr6 chr11 chr19 chr22; do
  echo "=== Subsetting $CHR (partial download) ==="
  grep "^${CHR}" snps_16_regions.bed > "${OUTDIR}/${CHR}.bed"

  # Download only the index (tiny) and the needed blocks via gsutil range
  gsutil -u "$PROJECT" cp "${PHASE_DIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz.tbi" "${OUTDIR}/"
  gsutil -u "$PROJECT" cp "${PHASE_DIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz" "${OUTDIR}/"

  # Now you have local files; use bcftools normally
  bcftools view \
    --regions-file "${OUTDIR}/${CHR}.bed" \
    --samples-file scd_participants_476.txt \
    --force-samples \
    -Oz -o "${OUTDIR}/${CHR}.476.17pos.vcf.gz"

  bcftools index -t "${OUTDIR}/${CHR}.476.16pos.vcf.gz"
  echo " Done ${CHR}"

  # Clean up to save disk
  rm -f "${OUTDIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz"*
done


In [ ]:
%%bash
set -euo pipefail

PROJECT="terra-vpc-sc-ce41bb61"
PHASE_DIR="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUTDIR="phased_extract_16"
mkdir -p "$OUTDIR"

# only these are safe on your current disk
for CHR in chr11 chr19 chr22; do
  echo "=== Subsetting $CHR ==="
  grep "^${CHR}" snps_16_regions.bed > "${OUTDIR}/${CHR}.bed"

  # download 1 chromosome + index
  gsutil -u "$PROJECT" cp "${PHASE_DIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz.tbi" "${OUTDIR}/"
  gsutil -u "$PROJECT" cp "${PHASE_DIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz" "${OUTDIR}/"

  # ✅ IMPORTANT: specify the input VCF at the end
  bcftools view \
    --regions-file "${OUTDIR}/${CHR}.bed" \
    --samples-file scd_participants_476.txt \
    --force-samples \
    -Oz -o "${OUTDIR}/${CHR}.476.16pos.vcf.gz" \
    "${OUTDIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz"

  bcftools index -t "${OUTDIR}/${CHR}.476.16pos.vcf.gz"
  echo "✅ Done ${CHR}"

  # delete the huge chromosome files immediately
  rm -f "${OUTDIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz" "${OUTDIR}/${CHR}_AOU_v8.2_allsamples_phased.vcf.gz.tbi"
  df -h | head -n 2
done


---
## Step 12 — Kill Stray Spark Processes (If Needed)

**What this does:**  
Cleans up any lingering Spark or Java processes from previous Hail runs  
that may interfere with a fresh Hail initialisation.

> 💡 *Run this cell if Hail fails to initialise with a port conflict error.*

In [ ]:
%%bash
set -euo pipefail
# kill any stray spark/yarn client processes
pkill -f spark || true
pkill -f yarn || true
pkill -f java || true


---
## Step 13 — Reinitialise Hail

**What this does:**  
Reinitialises Hail with the correct GRCh38 reference and requester-pays  
configuration. Run this after any Hail session interruption.

In [ ]:
import os, hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT = os.environ["GOOGLE_PROJECT"]

hl.init(
    default_reference="GRCh38",
    tmp_dir=f"{WORKSPACE_BUCKET}/hail_tmp",
    log=f"{WORKSPACE_BUCKET}/logs/hail_subset_16pos_476.log",
    idempotent=True,
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, []),
)

print("Hail started")
print("GOOGLE_PROJECT:", GOOGLE_PROJECT)
print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET)


---
## Step 14 — Check Disk Space

**What this does:**  
Verifies available local disk and HDFS space before running  
large VCF operations to avoid out-of-space errors.

In [ ]:
%%bash
set -euo pipefail
echo "Local disk:"
df -h | head -n 5
echo
echo "HDFS disk (if available):"
hdfs dfs -df -h 2>/dev/null || echo "No hdfs command / not on spark cluster yet"


---
## Step 15 — Hail Extraction: All Chromosomes (Full Run)

**What this does:**  
The complete Hail extraction for all chromosomes in the 17-position set.  
Uses requester-pays configuration to access `fc-aou-datasets-controlled`.

In [ ]:
import os, hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]

hl.init(
    default_reference="GRCh38",
    tmp_dir=f"{WORKSPACE_BUCKET}/hail_tmp",
    log=f"{WORKSPACE_BUCKET}/logs/hail_subset_16pos_476.log",
    idempotent=True,
    # requester pays: bill to your workspace project for this bucket
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, ["fc-aou-datasets-controlled"]),
)

print("✅ Hail started")
print("GOOGLE_PROJECT:", GOOGLE_PROJECT)
print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET)


In [ ]:
import os, hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]

PHASE_DIR = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUT = f"{WORKSPACE_BUCKET}/outputs/phased_16pos_476"

# init (Spark backend)
hl.init(
    default_reference="GRCh38",
    tmp_dir=f"{WORKSPACE_BUCKET}/hail_tmp",
    log=f"{WORKSPACE_BUCKET}/logs/hail_subset_16pos_476.log",
    idempotent=True,
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, ["fc-aou-datasets-controlled"]),
)

# 476 keep list
keep_path = f"{WORKSPACE_BUCKET}/inputs/scd_participants_iid.txt"
keep_ht = hl.import_table(keep_path, no_header=True).rename({'f0': 's'}).key_by('s')

# chr22 only (2 loci)
chrom = "chr22"
positions = [36265860, 36265988]

target_loci = hl.literal({hl.parse_locus(f"{chrom}:{p}", reference_genome="GRCh38") for p in positions})

vcf = f"{PHASE_DIR}/{chrom}_AOU_v8.2_allsamples_phased.vcf.gz"
print("Reading:", vcf)

mt = hl.import_vcf(vcf, reference_genome="GRCh38", force_bgz=True, min_partitions=64)

# keep only loci + your 476 samples
mt = mt.filter_rows(target_loci.contains(mt.locus))
mt = mt.filter_cols(hl.is_defined(keep_ht[mt.s]))

# keep only GT to make output smaller
mt = mt.select_entries("GT")

print("rows:", mt.count_rows(), "cols:", mt.count_cols())

hl.export_vcf(mt, f"{OUT}/{chrom}.476.2pos.phased.vcf.bgz", tabix=True)
print("Wrote:", f"{OUT}/{chrom}.476.2pos.phased.vcf.bgz")


---
## Step 16 — Validate Chr22 Output

**What this does:**  
Downloads the chr22 VCF subset, checks variant positions and  
queries genotypes for a sample to validate the extraction.

In [ ]:
%%bash
set -euo pipefail
OUT="${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr22.476.2pos.phased.vcf.bgz"

# copy small output locally (it’s tiny)
gsutil cp "$OUT" ./chr22.small.vcf.bgz
gsutil cp "$OUT.tbi" ./chr22.small.vcf.bgz.tbi

# show sample names (first 10)
bcftools query -l ./chr22.small.vcf.bgz | head


In [ ]:
%%bash
set -euo pipefail
bcftools query -f '%CHROM\t%POS\t%ID\t%REF\t%ALT\n' ./chr22.small.vcf.bgz


In [ ]:
%%bash
set -euo pipefail

# get first 10 sample IDs from the VCF
bcftools query -l ./chr22.small.vcf.bgz | head -n 10 > samples10.txt

echo "First 10 samples:"
cat samples10.txt

echo ""
echo "Genotypes for those 10 samples at the 2 SNPs:"
bcftools query \
  -S samples10.txt \
  -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' \
  ./chr22.small.vcf.bgz


In [ ]:
%%bash
set -euo pipefail
SAMPLE="1006406"
bcftools query \
  -s "$SAMPLE" \
  -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n' \
  ./chr22.small.vcf.bgz


---
## Step 17 — Build Chr22 Genotype Table

**What this does:**  
Converts the chr22 VCF into a wide-format pandas DataFrame  
(rows = variants, columns = sample IDs) and saves as CSV.

In [ ]:
import glob, pandas as pd, subprocess, shlex, os

# auto-detect local chr22 vcf
candidates = sorted(glob.glob("./chr22*.vcf.bgz"))
if not candidates:
    raise FileNotFoundError("No local chr22*.vcf.bgz found. Run gsutil cp again or check your directory.")
vcf_local = candidates[0]
print("Using:", vcf_local)

# samples
samples = subprocess.check_output(f"bcftools query -l {vcf_local}", shell=True, text=True).strip().splitlines()
print("Sample count:", len(samples))

# genotype lines (2 variants)
lines = subprocess.check_output(
    f"bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n' {vcf_local}",
    shell=True, text=True
).strip().splitlines()

rows = []
for line in lines:
    parts = line.split("\t")
    chrom, pos, ref, alt = parts[0], int(parts[1]), parts[2], parts[3]
    gts = parts[4:]
    row = {"CHROM": chrom, "POS": pos, "REF": ref, "ALT": alt}
    row.update({samples[i]: gts[i] for i in range(len(samples))})
    rows.append(row)

df_chr22 = pd.DataFrame(rows).sort_values("POS")
df_chr22


In [ ]:
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
OUTDIR = f"{WORKSPACE_BUCKET}/outputs/phased_16pos_476"

csv_name = "chr22_476_genotypes_table.csv"
df_chr22.to_csv(csv_name, index=False)

subprocess.run(shlex.split(f"gsutil cp {csv_name} {OUTDIR}/"), check=True)
print("✅ Uploaded:", f"{OUTDIR}/{csv_name}")


---
## Step 18 — Extract Chr19 (APOE) Genotypes

**What this does:**  
Specific extraction for chromosome 19 (APOE locus):
- Positions: `chr19:44908684` and `chr19:44908822` (APOE ε2/ε3/ε4 variants)

**Expected output:** Wide genotype table with 476 samples × 2 APOE positions.

> 🧬 *APOE genotype is relevant for stroke risk assessment in SCD patients.*

In [ ]:
%%bash
set -euo pipefail

cat > snps_16_regions.bed << 'EOF'
chr1    100718269   100718269
chr2    60487726    60487726
chr2    60490908    60490908
chr2    60498316    60498316
chr6    31575254    31575254
chr6    135097495   135097495
chr6    135097526   135097526
chr6    135097880   135097880
chr11   5239228     5239228
chr11   5242453     5242453
chr11   5248569     5248569
chr11   5254939     5254939
chr19   44908684    44908684
chr19   44908822    44908822
chr22   36265860    36265860
chr22   36265988    36265988
EOF

ls -lh snps_16_regions.bed
wc -l snps_16_regions.bed


In [ ]:
import os, hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]

PHASE_DIR = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUT      = f"{WORKSPACE_BUCKET}/outputs/phased_16pos_476"

hl.init(
    default_reference="GRCh38",
    tmp_dir=f"{WORKSPACE_BUCKET}/hail_tmp",
    log=f"{WORKSPACE_BUCKET}/logs/hail_subset_16pos_476.log",
    idempotent=True,
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, ["fc-aou-datasets-controlled"]),
)

keep_path = f"{WORKSPACE_BUCKET}/inputs/scd_participants_iid.txt"
keep_ht = hl.import_table(keep_path, no_header=True).rename({'f0': 's'}).key_by('s')

print("Ready. Bucket:", WORKSPACE_BUCKET)


In [ ]:
chrom = "chr19"
positions = [44908684, 44908822]  # APOE

vcf = f"{PHASE_DIR}/{chrom}_AOU_v8.2_allsamples_phased.vcf.gz"
print("Reading:", vcf)

target_loci = hl.literal({hl.parse_locus(f"{chrom}:{p}", reference_genome="GRCh38") for p in positions})

mt = hl.import_vcf(vcf, reference_genome="GRCh38", force_bgz=True, min_partitions=64)
mt = mt.filter_rows(target_loci.contains(mt.locus))
mt = mt.filter_cols(hl.is_defined(keep_ht[mt.s]))
mt = mt.select_entries("GT")

print("rows:", mt.count_rows(), "cols:", mt.count_cols())

out_vcf = f"{OUT}/{chrom}.476.{len(positions)}pos.phased.vcf.bgz"
hl.export_vcf(mt, out_vcf, tabix=True)
print("Wrote:", out_vcf)


---
## Step 19 — Process Chr19 VCF with bcftools

**What this does:**  
Downloads the chr19 VCF locally and uses bcftools to:  
1. Verify variant positions  
2. Export genotypes in wide-format TSV  
3. Convert to long format for downstream merging

In [ ]:
%%bash
set -euo pipefail
VCF="${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr19.476.2pos.phased.vcf.bgz"

gsutil cp "$VCF" ./chr19.476.2pos.vcf.bgz
gsutil cp "$VCF.tbi" ./chr19.476.2pos.vcf.bgz.tbi

echo "Samples in VCF:"
bcftools query -l ./chr19.476.2pos.vcf.bgz | wc -l

echo "Variant rows in VCF:"
bcftools view -H ./chr19.476.2pos.vcf.bgz | wc -l


In [ ]:
%%bash
set -euo pipefail
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' ./chr19.476.2pos.vcf.bgz | head -n 2 | cut -f1-19


In [ ]:
%%bash
set -euo pipefail
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' ./chr19.476.2pos.vcf.bgz > chr19_2snps_genotypes_wide.tsv
head -n 2 chr19_2snps_genotypes_wide.tsv | cut -f1-12


In [ ]:
%%bash
set -euo pipefail
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%SAMPLE=%GT]\n' ./chr19.476.2pos.vcf.bgz \
| awk 'BEGIN{OFS="\t"; print "CHROM","POS","REF","ALT","SAMPLE","GT"}
{
  chrom=$1; pos=$2; ref=$3; alt=$4;
  for(i=5;i<=NF;i++){
    split($i,a,"="); print chrom,pos,ref,alt,a[1],a[2]
  }
}' > chr19_2snps_genotypes_long.tsv

head chr19_2snps_genotypes_long.tsv


In [ ]:
%%bash
set -euo pipefail
SRC="${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr19.476.2pos.phased.vcf.bgz"
DEST_DIR="${WORKSPACE_BUCKET}/outputs/final_phased_16pos_476"

gsutil -m mkdir -p "$DEST_DIR" || true
gsutil -m cp "$SRC" "$SRC.tbi" "$DEST_DIR/"
gsutil ls -lh "$DEST_DIR" | grep chr19


In [ ]:
%%bash
set -euo pipefail
DEST_DIR="${WORKSPACE_BUCKET}/outputs/final_phased_16pos_476"

gsutil cp chr19_2snps_genotypes_wide.tsv "$DEST_DIR/" || true
gsutil cp chr19_2snps_genotypes_long.tsv "$DEST_DIR/" || true
gsutil ls -lh "$DEST_DIR" | grep chr19


---
## Step 20 — Extract Chr6 (TNF + HBS1L-MYB/HMIP) Genotypes

**What this does:**  
Hail extraction for chromosome 6 targeting:
- `chr6:31575254` — TNF locus (inflammatory modifier)  
- `chr6:135097495`, `135097526`, `135097880` — HBS1L-MYB/HMIP (HbF regulation)

**Note:** HMIP positions 135097495, 135097526, 135097880 were assessed but  
chr6:135097495 and 135097880 are excluded from the final 16-SNP set  
(confirmed by ACAF threshold check).

> 🧬 *HMIP is one of the three major HbF-regulating quantitative trait loci in SCD.*

In [ ]:
import os
import hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]

PHASE_DIR = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUT_DIR   = f"{WORKSPACE_BUCKET}/outputs/phased_16pos_476"
TMP_DIR   = f"{WORKSPACE_BUCKET}/hail_tmp"
LOG_PATH  = f"{WORKSPACE_BUCKET}/logs/hail_subset_chr6_476.log"

# Init Hail (Spark backend) + requester pays
hl.init(
    default_reference="GRCh38",
    tmp_dir=TMP_DIR,
    log=LOG_PATH,
    idempotent=True,
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, ["fc-aou-datasets-controlled"]),
)

# 476 keep list
keep_path = f"{WORKSPACE_BUCKET}/inputs/scd_participants_iid.txt"
keep_ht = hl.import_table(keep_path, no_header=True).rename({'f0': 's'}).key_by('s')

# chr6 positions from your BED
chrom = "chr6"
positions = [31575254, 135097495, 135097526, 135097880]

target_loci = hl.literal({
    hl.parse_locus(f"{chrom}:{p}", reference_genome="GRCh38")
    for p in positions
})

vcf_path = f"{PHASE_DIR}/{chrom}_AOU_v8.2_allsamples_phased.vcf.gz"
print("Reading:", vcf_path)

mt = hl.import_vcf(
    vcf_path,
    reference_genome="GRCh38",
    force_bgz=True,
    min_partitions=64
)

# Filter to your loci + your 476 samples
mt = mt.filter_rows(target_loci.contains(mt.locus))
mt = mt.filter_cols(hl.is_defined(keep_ht[mt.s]))

# Keep only genotype calls (smaller output)
mt = mt.select_entries("GT")

print("rows:", mt.count_rows(), "cols:", mt.count_cols())

out_vcf = f"{OUT_DIR}/{chrom}.476.4pos.phased.vcf.bgz"
hl.export_vcf(mt, out_vcf, tabix=True)

print(" Wrote:", out_vcf)


In [ ]:
import subprocess, shlex

VCF_GS = "${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr6.476.4pos.phased.vcf.bgz"

subprocess.run(shlex.split(f"gsutil cp {VCF_GS} ./chr6.vcf.bgz"), check=True)
subprocess.run(shlex.split(f"gsutil cp {VCF_GS}.tbi ./chr6.vcf.bgz.tbi"), check=True)

print("Downloaded chr6 VCF locally: chr6.vcf.bgz")


In [ ]:
import subprocess

print(subprocess.check_output(
    "bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' ./chr6.vcf.bgz",
    shell=True, text=True
))


In [ ]:
import pandas as pd
import subprocess

# sample names (should be 476)
samples = subprocess.check_output("bcftools query -l ./chr6.vcf.bgz", shell=True, text=True).strip().splitlines()
print("Number of samples:", len(samples))

# genotype table lines
txt = subprocess.check_output(
    "bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' ./chr6.vcf.bgz",
    shell=True, text=True
).strip().splitlines()

rows = []
for line in txt:
    parts = line.split("\t")
    chrom, pos, ref, alt = parts[0], int(parts[1]), parts[2], parts[3]
    gts = parts[4:]
    row = {"CHROM": chrom, "POS": pos, "REF": ref, "ALT": alt}
    row.update({samples[i]: gts[i] for i in range(len(samples))})
    rows.append(row)

df_chr6 = pd.DataFrame(rows)

# ✅ view table
df_chr6


In [ ]:
df_chr6.to_csv("chr6_476_genotypes_wide.csv", index=False)
print("Saved: chr6_476_genotypes_wide.csv")


In [ ]:
%%bash
set -euo pipefail
VCF_GS="${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr6.476.4pos.phased.vcf.bgz"
gsutil cp "$VCF_GS" ./chr6.vcf.bgz
gsutil cp "$VCF_GS.tbi" ./chr6.vcf.bgz.tbi


---
## Step 21 — Handle Multi-Allelic Sites in Chr6

**What this does:**  
Splits multi-allelic VCF records into separate biallelic records  
using `bcftools norm -m -both`, then specifically processes the  
HMIP `CC` allele at chr6:135097526.

In [ ]:
%%bash
set -euo pipefail

bcftools norm -m -both -Oz -o chr6.split.vcf.gz chr6.vcf.bgz
bcftools index -t chr6.split.vcf.gz

set +o pipefail
bcftools view -r chr6:135097526 chr6.split.vcf.gz | head -n 30
set -o pipefail


In [ ]:
%%bash
set -euo pipefail

# ALT = CC carriers (any sample where GT is not 0|0 or 0/0)
bcftools query -r chr6:135097526 -i 'ALT="CC"' \
  -f 'CC\t%CHROM\t%POS\t%REF\t%ALT[\t%SAMPLE:%GT]\n' chr6.split.vcf.gz \
| awk 'BEGIN{OFS="\t"}{
    # print only sample:GT pairs that are not 0|0 / 0/0 / missing
    out=$1"\t"$2"\t"$3"\t"$4"\t"$5
    for(i=6;i<=NF;i++){
      split($i,a,":"); gt=a[2]
      if(gt!="0|0" && gt!="0/0" && gt!="./." && gt!=".|.") out=out"\t"$i
    }
    print out
  }' > ALT_CC_carriers.tsv

# ALT = G carriers
bcftools query -r chr6:135097526 -i 'ALT="G"' \
  -f 'G\t%CHROM\t%POS\t%REF\t%ALT[\t%SAMPLE:%GT]\n' chr6.split.vcf.gz \
| awk 'BEGIN{OFS="\t"}{
    out=$1"\t"$2"\t"$3"\t"$4"\t"$5
    for(i=6;i<=NF;i++){
      split($i,a,":"); gt=a[2]
      if(gt!="0|0" && gt!="0/0" && gt!="./." && gt!=".|.") out=out"\t"$i
    }
    print out
  }' > ALT_G_carriers.tsv

echo "Saved:"
ls -lh ALT_CC_carriers.tsv ALT_G_carriers.tsv


In [ ]:
%%bash
set -euo pipefail
echo "CC carriers (first line):"
head -n 1 ALT_CC_carriers.tsv | cut -c1-250; echo "..."

echo "G carriers (first line):"
head -n 1 ALT_G_carriers.tsv | cut -c1-250; echo "..."


---
## Step 22 — Extract Chr11 (HBB-cluster + HBG2) Genotypes

**What this does:**  
Extraction for chromosome 11 — the beta-globin locus:
- `chr11:5239228`, `5242453`, `5248569` — HBB-cluster (SCD-causal variants)  
- `chr11:5254939` — HBG2 (foetal haemoglobin regulator)

**Note:** HBB positions have multi-allelic sites (A→C and A→T at chr11:5248569)  
which are processed separately before merging.

> 🧬 *The HBB cluster contains the causative SCD mutation (HbS: rs334)*

In [ ]:
import pandas as pd
import subprocess, shlex

VCF_GS = f"{WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr11.476.4pos.phased.vcf.bgz"

# download locally
subprocess.run(shlex.split(f"gsutil cp {VCF_GS} ./chr11.vcf.bgz"), check=True)
subprocess.run(shlex.split(f"gsutil cp {VCF_GS}.tbi ./chr11.vcf.bgz.tbi"), check=True)

# sample names
samples = subprocess.check_output("bcftools query -l ./chr11.vcf.bgz", shell=True, text=True).strip().splitlines()
print("Number of samples:", len(samples))

# genotype table
txt = subprocess.check_output(
    "bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' ./chr11.vcf.bgz",
    shell=True, text=True
).strip().splitlines()

rows = []
for line in txt:
    parts = line.split("\t")
    chrom, pos, ref, alt = parts[0], int(parts[1]), parts[2], parts[3]
    gts = parts[4:]
    row = {"CHROM": chrom, "POS": pos, "REF": ref, "ALT": alt}
    row.update({samples[i]: gts[i] for i in range(len(samples))})
    rows.append(row)

df_chr11 = pd.DataFrame(rows)
df_chr11


In [ ]:
%%bash
set -euo pipefail
bcftools view -r chr11:5248569 chr11.split.vcf.gz | sed -n '1,40p'


In [ ]:
%%bash
set -euo pipefail

VCF="chr11.split.vcf.gz"

# A -> C file
bcftools view -r chr11:5248569 -i 'ALT="C"' -Oz -o chr11_5248569_AtoC.vcf.gz "$VCF"
bcftools index -t chr11_5248569_AtoC.vcf.gz

# A -> T file
bcftools view -r chr11:5248569 -i 'ALT="T"' -Oz -o chr11_5248569_AtoT.vcf.gz "$VCF"
bcftools index -t chr11_5248569_AtoT.vcf.gz

ls -lh chr11_5248569_AtoC.vcf.gz* chr11_5248569_AtoT.vcf.gz*


In [ ]:
%%bash
set -euo pipefail

# get sample list once (same in both files)
bcftools query -l chr11_5248569_AtoC.vcf.gz > samples.txt

# GT per sample for each allele-specific VCF
bcftools query -f '[%GT\n]' chr11_5248569_AtoC.vcf.gz | tr '\t' '\n' | sed '/^$/d' > gt_AtoC.txt
bcftools query -f '[%GT\n]' chr11_5248569_AtoT.vcf.gz | tr '\t' '\n' | sed '/^$/d' > gt_AtoT.txt

paste samples.txt gt_AtoC.txt gt_AtoT.txt > chr11_5248569_AtoC_AtoT_table.tsv
printf "sample\tGT_AtoC\tGT_AtoT\n" | cat - chr11_5248569_AtoC_AtoT_table.tsv > chr11_5248569_table.tsv

head -n 15 chr11_5248569_table.tsv


In [ ]:
%%bash
set -euo pipefail

for VCF in chr11_5248569_AtoC.vcf.gz chr11_5248569_AtoT.vcf.gz; do
  base=$(basename "$VCF" .vcf.gz)

  # sample names (476 lines)
  bcftools query -l "$VCF" > ${base}.samples.txt

  # genotypes for the single variant (one line with 476 GTs)
  bcftools query -f '[%GT\n]' "$VCF" > ${base}.gtline.txt

  # make "one GT per line" (476 lines)
  tr '\t' '\n' < ${base}.gtline.txt > ${base}.gt.txt
  sed -i '/^$/d' ${base}.gt.txt

  # combine sample + GT
  paste ${base}.samples.txt ${base}.gt.txt > ${base}.sample_gt.tsv

  # keep only carriers (non-0|0)
  awk '($2!="0|0" && $2!="0/0" && $2!="./." && $2!=".|.")' \
    ${base}.sample_gt.tsv > carriers_${base}.tsv

  echo "✅ $VCF carriers count:"
  wc -l carriers_${base}.tsv
done

echo
echo "Preview A->C carriers:"
head carriers_chr11_5248569_AtoC.tsv

echo
echo "Preview A->T carriers:"
head carriers_chr11_5248569_AtoT.tsv


In [ ]:
df_chr11.to_csv("chr11_476_genotypes_wide.csv", index=False)
subprocess.run(shlex.split(f"gsutil cp chr11_476_genotypes_wide.csv {OUT_DIR}/"), check=True)
print("Saved + uploaded: chr11_476_genotypes_wide.csv")


---
## Step 23 — Extract Chr2 (BCL11A) Genotypes

**What this does:**  
Extraction for chromosome 2 — the BCL11A locus:
- `chr2:60487726`, `60490908`, `60498316` — BCL11A enhancer SNPs (HbF regulation)

> 🧬 *BCL11A is the primary transcriptional repressor of HbF production.  
> SNPs at this locus are strong predictors of HbF levels in SCD patients.*

In [ ]:
import os, hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]

PHASE_DIR = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUT_DIR   = f"{WORKSPACE_BUCKET}/outputs/phased_16pos_476"
TMP_DIR   = f"{WORKSPACE_BUCKET}/hail_tmp"
LOG_PATH  = f"{WORKSPACE_BUCKET}/logs/hail_subset_chr2_476.log"

hl.init(
    default_reference="GRCh38",
    tmp_dir=TMP_DIR,
    log=LOG_PATH,
    idempotent=True,
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, ["fc-aou-datasets-controlled"]),
)

keep_path = f"{WORKSPACE_BUCKET}/inputs/scd_participants_iid.txt"
keep_ht = hl.import_table(keep_path, no_header=True).rename({'f0':'s'}).key_by('s')

chrom = "chr2"
positions = [60487726, 60490908, 60498316]

target_loci = hl.literal({hl.parse_locus(f"{chrom}:{p}", reference_genome="GRCh38") for p in positions})

vcf_path = f"{PHASE_DIR}/{chrom}_AOU_v8.2_allsamples_phased.vcf.gz"
print("Reading:", vcf_path)

mt = hl.import_vcf(vcf_path, reference_genome="GRCh38", force_bgz=True, min_partitions=64)
mt = mt.filter_rows(target_loci.contains(mt.locus))
mt = mt.filter_cols(hl.is_defined(keep_ht[mt.s]))
mt = mt.select_entries("GT")

print("rows:", mt.count_rows(), "cols:", mt.count_cols())

out_vcf = f"{OUT_DIR}/{chrom}.476.3pos.phased.vcf.bgz"
hl.export_vcf(mt, out_vcf, tabix=True)
print("Wrote:", out_vcf)


In [ ]:
%%bash
set -euo pipefail
OUT="${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr2.476.3pos.phased.vcf.bgz"

gsutil cp "$OUT" ./chr2.vcf.bgz
gsutil cp "$OUT.tbi" ./chr2.vcf.bgz.tbi

echo "Sample count:"
bcftools query -l ./chr2.vcf.bgz | wc -l

echo "Variants:"
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' ./chr2.vcf.bgz


In [ ]:
import subprocess, pandas as pd, shlex, os

vcf_local = "./chr2.vcf.bgz"

# 1) all samples
samples = subprocess.check_output(
    f"bcftools query -l {vcf_local}",
    shell=True, text=True
).strip().splitlines()

print("Samples:", len(samples))

# 2) variant lines with genotypes
lines = subprocess.check_output(
    f"bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n' {vcf_local}",
    shell=True, text=True
).strip().splitlines()

long = []
for line in lines:
    parts = line.split("\t")
    chrom, pos, ref, alt = parts[0], int(parts[1]), parts[2], parts[3]
    gts = parts[4:]
    for i, s in enumerate(samples):
        long.append({"sample": s, "CHROM": chrom, "POS": pos, "REF": ref, "ALT": alt, "GT": gts[i]})

df_long = pd.DataFrame(long).sort_values(["CHROM","POS","sample"])
df_long.head(20)


In [ ]:
df_long.to_csv("chr2_476_3pos_long.tsv", sep="\t", index=False)
print("Saved: chr2_476_3pos_long.tsv")


In [ ]:
%%bash
set -euo pipefail

# input: your chr2 subset (local)
IN="./chr2.vcf.bgz"
OUT="./chr2.split.vcf.gz"

# split multi-allelics into separate biallelic records
bcftools norm -m -any -Oz -o "$OUT" "$IN"
bcftools index -t "$OUT"

# show the split records at that exact position
bcftools view -r chr2:60498316 "$OUT" | sed -n '1,60p'


In [ ]:
%%bash
set -euo pipefail

VCF="./chr2.split.vcf.gz"

# G -> A
bcftools view -r chr2:60498316 -i 'ALT="A"' -Oz -o chr2_60498316_GtoA.vcf.gz "$VCF"
bcftools index -t chr2_60498316_GtoA.vcf.gz

# G -> C
bcftools view -r chr2:60498316 -i 'ALT="C"' -Oz -o chr2_60498316_GtoC.vcf.gz "$VCF"
bcftools index -t chr2_60498316_GtoC.vcf.gz

ls -lh chr2_60498316_GtoA.vcf.gz* chr2_60498316_GtoC.vcf.gz*


In [ ]:
%%bash
set -euo pipefail

echo "G->A (one row, 476 GTs):"
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' chr2_60498316_GtoA.vcf.gz | head -n 1

echo ""
echo "G->C (one row, 476 GTs):"
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' chr2_60498316_GtoC.vcf.gz | head -n 1


---
## Step 24 — Extract Chr1 (VCAM1) Genotype

**What this does:**  
Extraction for chromosome 1:
- `chr1:100718269` — VCAM1 (Vascular Cell Adhesion Molecule 1)

> 🧬 *VCAM1 mediates red blood cell adhesion to the endothelium —  
> a key mechanism in vaso-occlusive crisis (VOC) pathophysiology in SCD.*

In [ ]:
import os, hail as hl

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]

PHASE_DIR = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/phasing"
OUT_DIR   = f"{WORKSPACE_BUCKET}/outputs/phased_16pos_476"
TMP_DIR   = f"{WORKSPACE_BUCKET}/hail_tmp"
LOG_PATH  = f"{WORKSPACE_BUCKET}/logs/hail_subset_chr1_476.log"

hl.init(
    default_reference="GRCh38",
    tmp_dir=TMP_DIR,
    log=LOG_PATH,
    idempotent=True,
    gcs_requester_pays_configuration=(GOOGLE_PROJECT, ["fc-aou-datasets-controlled"]),
)

keep_path = f"{WORKSPACE_BUCKET}/inputs/scd_participants_iid.txt"
keep_ht = hl.import_table(keep_path, no_header=True).rename({'f0':'s'}).key_by('s')

chrom = "chr1"
positions = [100718269]

target_loci = hl.literal({hl.parse_locus(f"{chrom}:{p}", reference_genome="GRCh38") for p in positions})

vcf_path = f"{PHASE_DIR}/{chrom}_AOU_v8.2_allsamples_phased.vcf.gz"
print("Reading:", vcf_path)

mt = hl.import_vcf(vcf_path, reference_genome="GRCh38", force_bgz=True, min_partitions=64)
mt = mt.filter_rows(target_loci.contains(mt.locus))
mt = mt.filter_cols(hl.is_defined(keep_ht[mt.s]))
mt = mt.select_entries("GT")

print("rows:", mt.count_rows(), "cols:", mt.count_cols())

out_vcf = f"{OUT_DIR}/{chrom}.476.1pos.phased.vcf.bgz"
hl.export_vcf(mt, out_vcf, tabix=True)
print("Wrote:", out_vcf)


In [ ]:
 %%bash
set -euo pipefail

VCF_GS="${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr1.476.1pos.phased.vcf.bgz"

gsutil cp "$VCF_GS" ./chr1.vcf.bgz
gsutil cp "$VCF_GS.tbi" ./chr1.vcf.bgz.tbi

echo "Sample count:"
bcftools query -l ./chr1.vcf.bgz | wc -l

echo "Variant info:"
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' ./chr1.vcf.bgz


In [ ]:
%%bash
set -euo pipefail

bcftools query -l ./chr1.vcf.bgz | head -n 10 > chr1.samples10.txt

echo "First 10 samples:"
cat chr1.samples10.txt

echo ""
echo "Genotypes for those 10:"
bcftools query -S chr1.samples10.txt \
  -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' \
  ./chr1.vcf.bgz


In [ ]:
%%bash
set -euo pipefail

# Header: CHROM POS REF ALT + 476 sample IDs
{
  printf "CHROM\tPOS\tREF\tALT"
  bcftools query -l ./chr1.vcf.bgz | awk '{printf "\t%s",$0} END{printf "\n"}'

  # Data row(s): 1 variant × 476 genotypes
  bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n' ./chr1.vcf.bgz
} > chr1_476_gt_table.tsv

echo "Preview (first row, truncated):"
head -n 2 chr1_476_gt_table.tsv | cut -c1-200

echo "File size:"
ls -lh chr1_476_gt_table.tsv


In [ ]:
%%bash
set -euo pipefail

gsutil cp chr1_476_gt_table.tsv \
  "${WORKSPACE_BUCKET}/outputs/phased_16pos_476/"

echo "✅ Uploaded:"
echo "${WORKSPACE_BUCKET}/outputs/phased_16pos_476/chr1_476_gt_table.tsv"


---
## Step 25 — Merge All Chromosomes into Final VCF

**What this does:**  
Uses `bcftools` to:
1. Verify all per-chromosome VCF files are present  
2. Concatenate all chromosomes into a single merged VCF  
3. Deduplicate exact duplicate records  
4. Rename the final file to `FINAL_MERGED_ALL_476_16pos.phased.vcf.gz`

**Expected output:**  
- 476 samples  
- 16 variants  
- All chromosomes represented

In [ ]:
%%bash
set -euo pipefail

for f in chr1.vcf.bgz chr2.vcf.bgz chr6.vcf.bgz chr11.vcf.bgz chr19.476.2pos.vcf.bgz chr22.small.vcf.bgz; do
  if [ -f "$f" ]; then
    out="${f%.vcf.bgz}.split.vcf.gz"
    echo "Splitting $f -> $out"
    bcftools norm -m -any -Oz -o "$out" "$f"
    bcftools index -t "$out"
  fi
done

ls -lh *.split.vcf.gz* || true


In [ ]:
%%bash
set -euo pipefail

# 1) Remove old chr22 2pos if it exists locally (this avoids duplicates)
rm -f chr22.476.2pos.phased.vcf.bgz chr22.476.2pos.phased.vcf.bgz.tbi

# 2) Create FINAL merged file
FINAL_VCF="FINAL_MERGED_ALL_476_17pos.phased.vcf.gz"

rm -f "$FINAL_VCF" "$FINAL_VCF.tbi"

bcftools concat -a -Oz -o "$FINAL_VCF" \
  chr1*.vcf.bgz chr2*.vcf.bgz chr6*.vcf.bgz chr11*.vcf.bgz chr19*.vcf.bgz chr22.476.3pos.vcf.bgz

bcftools index -t "$FINAL_VCF"

# 3) Quick checks (should be 476 samples, 17 variants)
echo "Samples:";  bcftools query -l "$FINAL_VCF" | wc -l
echo "Variants:"; bcftools view -H "$FINAL_VCF" | wc -l
echo "chr22 loci:"; bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' -r chr22 "$FINAL_VCF"


In [ ]:
%%bash
set -euo pipefail

IN="FINAL_MERGED_ALL_476_17pos.phased.vcf.gz"
OUT="FINAL_MERGED_ALL_476_17pos.phased.DEDUP.vcf.gz"

bcftools norm -d exact -Oz -o "$OUT" "$IN"
bcftools index -t "$OUT"

echo "Samples:";  bcftools query -l "$OUT" | wc -l
echo "Variants:"; bcftools view -H "$OUT" | wc -l
echo "chr22 loci:"; bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' -r chr22 "$OUT"


In [ ]:
%%bash
set -euo pipefail
VCF="FINAL_MERGED_ALL_476_17pos.phased.DEDUP.vcf.gz"
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' "$VCF" | sort -V


In [ ]:
%%bash
set -euo pipefail

cp -f FINAL_MERGED_ALL_476_17pos.phased.DEDUP.vcf.gz FINAL_MERGED_ALL_476_16pos.phased.vcf.gz
cp -f FINAL_MERGED_ALL_476_17pos.phased.DEDUP.vcf.gz.tbi FINAL_MERGED_ALL_476_16pos.phased.vcf.gz.tbi

echo "Samples:";  bcftools query -l FINAL_MERGED_ALL_476_16pos.phased.vcf.gz | wc -l
echo "Variants:"; bcftools view -H FINAL_MERGED_ALL_476_16pos.phased.vcf.gz | wc -l


In [ ]:
%%bash
set -euo pipefail
ls -lh FINAL_MERGED_ALL_476_16pos.phased.vcf.gz FINAL_MERGED_ALL_476_16pos.phased.vcf.gz.tbi


In [ ]:
%%bash
set -euo pipefail
VCF="FINAL_MERGED_ALL_476_16pos.phased.vcf.gz"

echo "Samples:";  bcftools query -l "$VCF" | wc -l
echo "Variants:"; bcftools view -H "$VCF" | wc -l


In [ ]:
%%bash
set -euo pipefail
VCF="FINAL_MERGED_ALL_476_16pos.phased.vcf.gz"
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' "$VCF" | sort -V


In [ ]:
%%bash
set -euo pipefail
VCF="FINAL_MERGED_ALL_476_16pos.phased.vcf.gz"

# show first 5 variant lines (ignore SIGPIPE/broken pipe)
( bcftools view -H "$VCF" | head -n 5 ) || true


---
## Step 26 — Build Combined Genotype Table (All Chromosomes)

**What this does:**  
Reads all per-chromosome genotype CSVs and merges them into:
- **Wide format** — `ALL_476_genotypes_wide.csv` (476 rows × 16 variant columns)  
- **Long format** — `ALL_476_genotypes_long.csv` (one row per sample × variant)

**Expected output:** One row per participant, one column per SNP.

In [ ]:
import subprocess, pandas as pd, numpy as np, glob, os

def bcftools_samples(vcf):
    return subprocess.check_output(f"bcftools query -l {vcf}", shell=True, text=True).strip().splitlines()

def bcftools_wide_gt(vcf):
    samples = bcftools_samples(vcf)
    lines = subprocess.check_output(
        f"bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n' {vcf}",
        shell=True, text=True
    ).strip().splitlines()

    rows = []
    for line in lines:
        parts = line.split("\t")
        chrom, pos, ref, alt = parts[0], int(parts[1]), parts[2], parts[3]
        gts = parts[4:]
        row = {"CHROM": chrom, "POS": pos, "REF": ref, "ALT": alt}
        row.update({samples[i]: gts[i] for i in range(len(samples))})
        rows.append(row)

    wide = pd.DataFrame(rows).sort_values(["CHROM","POS","ALT"]).reset_index(drop=True)
    return wide, samples

def gt_to_haps(gt, ref, alt):
    """
    Convert phased GT like 0|1 or 1|0 or 1|1 into haplotype alleles.
    If unphased 0/1, we still map but hap order is unknown.
    """
    if gt is None or gt in ("./.", ".|.", "."):
        return (np.nan, np.nan)

    sep = "|" if "|" in gt else ("/" if "/" in gt else None)
    if sep is None:
        return (np.nan, np.nan)

    a, b = gt.split(sep)
    # Only handle diploid numeric GTs (0/1/2 etc.)
    try:
        a = int(a); b = int(b)
    except:
        return (np.nan, np.nan)

    # After splitting with bcftools norm -m -any, ALT is single allele → ALT index is 1
    # so genotype should be 0 or 1 for this row.
    def allele(idx):
        if idx == 0: return ref
        if idx == 1: return alt
        return f"ALT{idx}"   # safety

    return (allele(a), allele(b))

def make_long_and_summary(vcf):
    wide, samples = bcftools_wide_gt(vcf)

    # long format
    long = wide.melt(
        id_vars=["CHROM","POS","REF","ALT"],
        value_vars=samples,
        var_name="SAMPLE",
        value_name="GT"
    )

    # add haplotype alleles
    haps = long.apply(lambda r: gt_to_haps(r["GT"], r["REF"], r["ALT"]), axis=1)
    long["HAP1_ALLELE"] = [x[0] for x in haps]
    long["HAP2_ALLELE"] = [x[1] for x in haps]

    # variant-level summary
    def summarize_variant(dfv):
        gts = dfv["GT"].astype(str)
        called = ~gts.isin(["./.", ".|.", "."])
        n_called = int(called.sum())

        # carrier = anything not ref/ref (works after splitting; ref genotype is 0|0 or 0/0)
        carrier = called & ~gts.isin(["0|0","0/0"])
        n_carrier = int(carrier.sum())

        # haplotype ALT count (after split, ALT index is 1 -> genotype strings contain "1")
        # Count number of '1' alleles across both haps for called samples.
        # Example: 0|1 -> 1, 1|1 -> 2
        alt_haps = 0
        for gt in gts[called]:
            if "|" in gt:
                a,b = gt.split("|")
            elif "/" in gt:
                a,b = gt.split("/")
            else:
                continue
            alt_haps += (a=="1") + (b=="1")

        af = (alt_haps / (2*n_called)) if n_called>0 else np.nan

        # haplotype pattern counts (0|1 vs 1|0 etc.)
        hap_counts = gts[called].value_counts().to_dict()

        return pd.Series({
            "N_SAMPLES": len(dfv),
            "N_CALLED": n_called,
            "N_CARRIERS": n_carrier,
            "ALT_HAP_COUNT": int(alt_haps),
            "ALT_AF": af,
            "GT_COUNTS": str(hap_counts)
        })

    summary = long.groupby(["CHROM","POS","REF","ALT"], as_index=False).apply(summarize_variant)
    # pandas sometimes adds extra index; clean it
    summary = summary.reset_index(drop=True)

    return wide, long, summary

# Use split vcfs if available (best), otherwise fallback to original
vcfs = sorted(glob.glob("*.split.vcf.gz"))
if not vcfs:
    vcfs = [f for f in ["chr1.vcf.bgz","chr2.vcf.bgz","chr6.vcf.bgz","chr11.vcf.bgz","chr19.476.2pos.vcf.bgz","chr22.small.vcf.bgz"] if os.path.exists(f)]

print("VCFs found:", vcfs)

all_long = []
all_summary = []
all_wide = []

for vcf in vcfs:
    wide, long, summary = make_long_and_summary(vcf)
    wide["SOURCE_VCF"] = vcf
    long["SOURCE_VCF"] = vcf
    summary["SOURCE_VCF"] = vcf
    all_wide.append(wide)
    all_long.append(long)
    all_summary.append(summary)

wide_all = pd.concat(all_wide, ignore_index=True)
long_all = pd.concat(all_long, ignore_index=True)
summary_all = pd.concat(all_summary, ignore_index=True)

# Save outputs
wide_all.to_csv("ALL_476_genotypes_wide.csv", index=False)
long_all.to_csv("ALL_476_genotypes_long.csv", index=False)
summary_all.to_csv("ALL_variants_summary.csv", index=False)

print("Saved:")
print(" - ALL_476_genotypes_wide.csv")
print(" - ALL_476_genotypes_long.csv")
print(" - ALL_variants_summary.csv")

summary_all


---
## Step 27 — Compute Allele Frequency Summary Statistics

**What this does:**  
Parses the genotype counts per variant and computes:
- `N_0_0`, `N_0_1`, `N_1_0`, `N_1_1` — phased genotype counts  
- `N_CARRIERS_anyALT` — number of participants with at least one ALT allele  
- `ALT_AF_allelefreq` — alternative allele frequency in the SCD cohort  
- `ALT_HAP_COUNT` — haplotype-level ALT allele count

In [ ]:
import pandas as pd
import ast

# Change this to your real summary file name
# (the one that contains GT_COUNTS, ALT_AF, ALT_HAP_COUNT, etc.)
df = pd.read_csv("ALL_variants_summary.csv")

def parse_counts(x):
    # handles string dicts like "{'0|0': 329, '0|1': 73, ...}"
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(str(x))
    except Exception:
        return {}

counts = df["GT_COUNTS"].apply(parse_counts)

df["N_0_0"] = counts.apply(lambda d: d.get("0|0", 0) + d.get("0/0", 0))
df["N_0_1"] = counts.apply(lambda d: d.get("0|1", 0) + d.get("0/1", 0))
df["N_1_0"] = counts.apply(lambda d: d.get("1|0", 0) + d.get("1/0", 0))
df["N_1_1"] = counts.apply(lambda d: d.get("1|1", 0) + d.get("1/1", 0))

# Rename to clearer column names (only if these columns exist)
rename_map = {
    "N_CARRIERS": "N_CARRIERS_anyALT",
    "ALT_HAP_COUNT": "N_ALT_HAPLOTYPES",
    "ALT_AF": "ALT_AF_allelefreq",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

# Pick columns that exist (prevents KeyError)
want_cols = [
    "CHROM","POS","REF","ALT",
    "N_CALLED",
    "N_CARRIERS_anyALT",
    "N_ALT_HAPLOTYPES",
    "ALT_AF_allelefreq",
    "N_0_0","N_0_1","N_1_0","N_1_1",
    "SOURCE_VCF"
]
want_cols = [c for c in want_cols if c in df.columns]

pretty = df[want_cols].sort_values(["CHROM","POS","ALT"], na_position="last")

pretty.to_csv("ALL_variants_summary_pretty.csv", index=False)
pretty


In [ ]:
import pandas as pd

# If you already have this table as a CSV, load it:
# df = pd.read_csv("ALL_variants_summary_pretty.csv")

# If df is already in memory (like your printed table), skip the line above.

df["HAP1_ALT"] = df["N_1_0"] + df["N_1_1"]
df["HAP2_ALT"] = df["N_0_1"] + df["N_1_1"]

df["HAP1_only_carriers"] = df["N_1_0"]
df["HAP2_only_carriers"] = df["N_0_1"]
df["Both_haps_ALT"]      = df["N_1_1"]

# sanity check (for biallelic variants after splitting)
df["ALT_HAP_check"] = (df["HAP1_ALT"] + df["HAP2_ALT"]) - df["N_ALT_HAPLOTYPES"]

# nicer ordering
cols = [
    "CHROM","POS","REF","ALT",
    "N_CALLED","N_CARRIERS_anyALT","N_ALT_HAPLOTYPES","ALT_AF_allelefreq",
    "N_0_0","N_0_1","N_1_0","N_1_1",
    "HAP1_ALT","HAP2_ALT","HAP1_only_carriers","HAP2_only_carriers","Both_haps_ALT",
    "ALT_HAP_check",
    "SOURCE_VCF"
]
cols = [c for c in cols if c in df.columns]

df2 = df[cols].sort_values(["CHROM","POS","ALT"])
df2.to_csv("ALL_variants_summary_with_haplotypes.csv", index=False)

df2


---
## Step 28 — Map Variants to Genes & Build Gene-Level Summary

**What this does:**  
Adds gene name annotations to every variant using the position → gene map,  
then creates a gene-level summary table showing:
- Number of variants per gene  
- Total carriers per gene  
- Combined allele frequency per gene

In [ ]:
import pandas as pd

# df2 = your summary table (the one you printed)
# example: df2 = pd.read_csv("ALL_variants_summary_with_haplotypes.csv")

gene_map = {
    ("chr19", 44908684): "APOE",
    ("chr19", 44908822): "APOE",
    ("chr22", 36265860): "APOL1",
    ("chr22", 36265988): "APOL1",
    ("chr2",  60487726): "BCL11A",
    ("chr2",  60490908): "BCL11A",
    ("chr2",  60498316): "BCL11A",
    ("chr11", 5239228):  "HBB-cluster",
    ("chr11", 5242453):  "HBB-cluster",
    ("chr11", 5248569):  "HBB-cluster",
    ("chr11", 5254939):  "HBG2",
    ("chr6",  31575254): "TNF",
    ("chr1", 100718269): "VCAM1",
}

# Ensure POS is integer (important for matching)
df2["POS"] = df2["POS"].astype(int)

# Add gene column
df2["GENE"] = df2.apply(lambda r: gene_map.get((r["CHROM"], r["POS"]), "UNKNOWN"), axis=1)

# Put GENE near the front
front = ["CHROM", "POS", "GENE", "REF", "ALT"]
rest = [c for c in df2.columns if c not in front]
df2 = df2[front + rest]

# Save
df2.to_csv("ALL_variants_summary_with_haplotypes_and_genes.csv", index=False)

df2


In [ ]:
import pandas as pd

df2 = pd.read_csv("ALL_variants_summary_with_haplotypes_and_genes.csv")
df2


In [ ]:
df2 = pd.read_csv("ALL_variants_summary_with_haplotypes.csv")
df2


In [ ]:
gene_map = {
    ("chr19", 44908684): "APOE",
    ("chr19", 44908822): "APOE",
    ("chr22", 36265860): "APOL1",
    ("chr22", 36265988): "APOL1",
    ("chr2",  60487726): "BCL11A",
    ("chr2",  60490908): "BCL11A",
    ("chr2",  60498316): "BCL11A",
    ("chr11", 5239228):  "HBB-cluster",
    ("chr11", 5242453):  "HBB-cluster",
    ("chr11", 5248569):  "HBB-cluster",
    ("chr11", 5254939):  "HBG2",
    ("chr6",  31575254): "TNF",
    ("chr6",  135097526): "HBS1L-MYB (HMIP)",
    ("chr6",  135097880): "HBS1L-MYB (HMIP)",
    ("chr1", 100718269): "VCAM1",
}


In [ ]:
df2 = pd.read_csv("ALL_variants_summary_with_haplotypes_and_genes.csv")
df2["POS"] = df2["POS"].astype(int)

df2["GENE"] = df2.apply(lambda r: gene_map.get((r["CHROM"], r["POS"]), "UNKNOWN"), axis=1)

# keep GENE near front
front = ["CHROM","POS","GENE","REF","ALT"]
rest = [c for c in df2.columns if c not in front]
df2 = df2[front + rest]

df2.to_csv("ALL_variants_summary_with_haplotypes_and_genes_FIXED.csv", index=False)
df2


In [ ]:
gene_summary = (
    df2.groupby("GENE", as_index=False)
       .agg(
           N_variants=("POS","count"),
           Total_carriers=("N_CARRIERS_anyALT","sum"),
           Total_ALT_haplotypes=("N_ALT_HAPLOTYPES","sum")
       )
       .sort_values("Total_carriers", ascending=False)
)

gene_summary.to_csv("GENE_level_summary.csv", index=False)
gene_summary


In [ ]:
nice_cols = [
    "CHROM","POS","GENE","REF","ALT",
    "N_CALLED","N_CARRIERS_anyALT","ALT_AF_allelefreq",
    "N_0_0","N_0_1","N_1_0","N_1_1",
    "HAP1_only_carriers","HAP2_only_carriers","Both_haps_ALT",
    "SOURCE_VCF"
]
nice_cols = [c for c in nice_cols if c in df2.columns]

final_table = df2[nice_cols].sort_values(["GENE","CHROM","POS","ALT"])
final_table.to_csv("FINAL_SCD_476_16pos_table.csv", index=False)
final_table


---
## Step 29 — Visualise Genotype Distributions

**What this does:**  
Creates visualisation charts for each gene/variant showing:
- Genotype frequency bar charts (0|0, 0|1, 1|0, 1|1)  
- ALT allele dosage distributions across the SCD cohort  
- Gene-level carrier summary heatmap

**Expected output:** One chart per gene group.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load long genotypes (you already created this)
long_df = pd.read_csv("ALL_476_genotypes_long.csv")

# Build a nice variant label
def variant_label(r):
    return f"{r['CHROM']}:{int(r['POS'])} {r['REF']}>{r['ALT']}"

long_df["VARIANT"] = long_df.apply(variant_label, axis=1)

# Convert GT -> dosage (ALT copies)
def gt_to_dosage(gt):
    if pd.isna(gt):
        return np.nan
    gt = str(gt)
    if gt in ("./.", ".|.", "."):
        return np.nan
    sep = "|" if "|" in gt else ("/" if "/" in gt else None)
    if sep is None:
        return np.nan
    a, b = gt.split(sep)
    if not (a.isdigit() and b.isdigit()):
        return np.nan
    return (a == "1") + (b == "1")   # after splitting, ALT index is 1

long_df["DOSAGE"] = long_df["GT"].apply(gt_to_dosage)

# Pivot to matrix: rows=variants, cols=samples
mat = long_df.pivot_table(index="VARIANT", columns="SAMPLE", values="DOSAGE", aggfunc="first")

# Optional: sort samples by total ALT dosage
sample_order = mat.sum(axis=0).sort_values(ascending=False).index
mat = mat[sample_order]

# Plot
plt.figure(figsize=(16, 6))
plt.imshow(mat.values, aspect="auto")  # do not set colors
plt.colorbar(label="ALT dosage (0/1/2)")
plt.yticks(range(len(mat.index)), mat.index, fontsize=8)

# Too many samples to label: leave x tick labels off
plt.xticks([])
plt.title("SCD cohort (n=476): ALT dosage heatmap across 16 positions (split variants)")
plt.xlabel("Samples (sorted by total ALT dosage)")
plt.ylabel("Variant")
plt.tight_layout()
plt.show()


In [ ]:
gene_map = {
    ("chr19", 44908684): "APOE",
    ("chr19", 44908822): "APOE",
    ("chr22", 36265860): "APOL1",
    ("chr22", 36265988): "APOL1",
    ("chr2",  60487726): "BCL11A",
    ("chr2",  60490908): "BCL11A",
    ("chr2",  60498316): "BCL11A",
    ("chr11", 5239228):  "HBB-cluster",
    ("chr11", 5242453):  "HBB-cluster",
    ("chr11", 5248569):  "HBB-cluster",
    ("chr11", 5254939):  "HBG2",
    ("chr6",  31575254): "TNF",
    ("chr6",  135097526): "HBS1L-MYB (HMIP)",
    ("chr6",  135097880): "HBS1L-MYB (HMIP)",
    ("chr1", 100718269): "VCAM1",
}

long_df["GENE"] = long_df.apply(lambda r: gene_map.get((r["CHROM"], int(r["POS"])), "UNKNOWN"), axis=1)
long_df["VARIANT_GENE"] = long_df.apply(lambda r: f"{r['GENE']} | {r['CHROM']}:{int(r['POS'])} {r['REF']}>{r['ALT']}", axis=1)

mat = long_df.pivot_table(index="VARIANT_GENE", columns="SAMPLE", values="DOSAGE", aggfunc="first")
sample_order = mat.sum(axis=0).sort_values(ascending=False).index
mat = mat[sample_order]

plt.figure(figsize=(16, 7))
plt.imshow(mat.values, aspect="auto")
plt.colorbar(label="ALT dosage (0/1/2)")
plt.yticks(range(len(mat.index)), mat.index, fontsize=8)
plt.xticks([])
plt.title("ALT dosage heatmap (Variant rows labelled by gene)")
plt.xlabel("Samples (sorted)")
plt.ylabel("Variant")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- 1) Load long format ----
long_df = pd.read_csv("ALL_476_genotypes_long.csv")

# ---- 2) Map gene name (edit if you want) ----
gene_map = {
    ("chr19", 44908684): "APOE",
    ("chr19", 44908822): "APOE",
    ("chr22", 36265860): "APOL1",
    ("chr22", 36265988): "APOL1",
    ("chr2",  60487726): "BCL11A",
    ("chr2",  60490908): "BCL11A",
    ("chr2",  60498316): "BCL11A",
    ("chr11", 5239228):  "HBB-cluster",
    ("chr11", 5242453):  "HBB-cluster",
    ("chr11", 5248569):  "HBB-cluster",
    ("chr11", 5254939):  "HBG2",
    ("chr6",  31575254): "TNF",
    ("chr6",  135097526): "HBS1L-MYB (HMIP)",
    ("chr6",  135097880): "HBS1L-MYB (HMIP)",
    ("chr1", 100718269): "VCAM1",
}

long_df["POS"] = long_df["POS"].astype(int)
long_df["GENE"] = long_df.apply(lambda r: gene_map.get((r["CHROM"], r["POS"]), "UNKNOWN"), axis=1)

# ---- 3) Convert GT -> ALT dosage (0/1/2) ----
def gt_to_dosage(gt):
    if pd.isna(gt):
        return np.nan
    gt = str(gt)
    if gt in ("./.", ".|.", "."):
        return np.nan
    sep = "|" if "|" in gt else ("/" if "/" in gt else None)
    if sep is None:
        return np.nan
    a, b = gt.split(sep)
    if not (a.isdigit() and b.isdigit()):
        return np.nan
    # After bcftools split (-m -any), ALT index is 1 for the row
    return (a == "1") + (b == "1")

long_df["DOSAGE"] = long_df["GT"].apply(gt_to_dosage)

# ---- 4) Build gene-level matrix: GENE x SAMPLE ----
gene_mat = (
    long_df
    .pivot_table(index="GENE", columns="SAMPLE", values="DOSAGE", aggfunc="sum")  # sum across variants in that gene
    .fillna(0)
)

# Optional: drop UNKNOWN if you want
# gene_mat = gene_mat.drop(index=["UNKNOWN"], errors="ignore")

# ---- 5) Sort samples for readability ----
# sort by total burden across all genes (descending)
sample_order = gene_mat.sum(axis=0).sort_values(ascending=False).index
gene_mat = gene_mat[sample_order]

# sort genes by total dosage (descending)
gene_order = gene_mat.sum(axis=1).sort_values(ascending=False).index
gene_mat = gene_mat.loc[gene_order]

# ---- 6) Save matrix ----
gene_mat.to_csv("SCD476_gene_dosage_matrix.csv")
print("Saved: SCD476_gene_dosage_matrix.csv")
print("Shape:", gene_mat.shape)

# ---- 7) Plot heatmap ----
plt.figure(figsize=(18, 5))
plt.imshow(gene_mat.values, aspect="auto")
plt.colorbar(label="Total ALT dosage (sum across variants)")
plt.yticks(range(len(gene_mat.index)), gene_mat.index, fontsize=10)
plt.xticks([])  # too many samples to label
plt.title("Gene-level ALT dosage heatmap (n=476)")
plt.xlabel("Samples (sorted by total dosage)")
plt.ylabel("Gene")
plt.tight_layout()
plt.show()


In [ ]:
# gene_mat already created above

gene_summary = pd.DataFrame({
    "GENE": gene_mat.index,
    "N_SAMPLES": gene_mat.shape[1],
    "N_CARRIERS_anyALT": (gene_mat > 0).sum(axis=1).astype(int),
    "MEAN_DOSAGE": gene_mat.mean(axis=1),
    "MEDIAN_DOSAGE": gene_mat.median(axis=1),
    "MAX_DOSAGE": gene_mat.max(axis=1),
    "TOTAL_DOSAGE": gene_mat.sum(axis=1),
}).sort_values(["N_CARRIERS_anyALT", "TOTAL_DOSAGE"], ascending=False)

gene_summary.to_csv("SCD476_gene_summary.csv", index=False)
print("Saved: SCD476_gene_summary.csv")

gene_summary


---
## ✅ Notebook 04 Complete — Genotypic Extraction Finished!

**Output files created:**

| File | Description |
|------|-------------|
| `FINAL_MERGED_ALL_476_16pos.phased.vcf.gz` | Final merged phased VCF — 476 samples, 16 SNPs |
| `FINAL_MERGED_ALL_476_16pos.phased.vcf.gz.tbi` | VCF index file |
| `ALL_476_genotypes_long.csv` | Long-format genotype table (per sample × variant) |
| `ALL_476_genotypes_wide.csv` | Wide-format genotype table (476 rows × 16 SNP columns) |
| `ALL_variants_summary_with_haplotypes_and_genes.csv` | Full summary with allele frequencies, haplotypes and gene annotations |
| Per-chromosome CSVs | `chr1_`, `chr2_`, `chr6_`, `chr11_`, `chr19_`, `chr22_` genotype tables |

---

**The complete SCD multi-modal pipeline is now finished:**

| Notebook | Topic | Status |
|---|---|---|
| 01 | EHR Demographics | ✅ |
| 02 | Phenotypic Complications | ✅ |
| 03 | Lab Measurements | ✅ |
| 04 | Genotypic WGS | ✅ |

---
> 💡 **Next steps:**  
> - Merge genotype data with `SCD_Demographics_with_Complications.csv` for genotype–phenotype analysis  
> - Use `ALL_476_genotypes_wide.csv` + `SCD_Complication_Severity_Only.csv` for ML models  
> - Cross-reference APOL1 G1/G2 risk variants with kidney complication data from Notebook 02